# Download FLUX.2 and NVARC to Google Drive

This notebook downloads the complete `black-forest-labs/FLUX.2-klein-base-4B` Hugging Face snapshot, the standalone BFL FLUX.2 autoencoder, and the published NVARC checkpoint from Kaggle into persistent Google Drive storage.

**Audience:** Diffusion-ARC development in the Google Colab web interface.

**Prerequisites:**

- A Google Drive with about 32 GB of free space.
- A Hugging Face account with access to `black-forest-labs/FLUX.2-dev` and a read token.
- A Kaggle account and API token.
- A Google Colab web runtime. Choose a CPU runtime; a GPU is not required for downloading.

**Run this notebook in Colab Web:**

1. Open [Google Colab](https://colab.research.google.com/) and choose **File > Upload notebook**.
2. Open **Runtime > Change runtime type** and select **CPU**.
3. Open the key icon (**Secrets**) in the left sidebar. Add `HF_TOKEN` and `KAGGLE_API_TOKEN`, and enable notebook access for both.
4. Choose **Runtime > Run all** and approve the Google Drive mount prompt.

**Result:** all checkpoint files remain in Drive after the Colab VM is recycled. Tokens are read from Colab Secrets and copied only into the current runtime environment; they are never written to this notebook, Drive, or a `.env` file.


## Outline

1. Install the download-only dependencies.
2. Mount Drive and configure persistent paths.
3. Load the Hugging Face and Kaggle tokens from Colab Secrets.
4. Download the complete FLUX snapshot and standalone VAE.
5. Download the pinned NVARC model version from Kaggle.
6. Validate checkpoint structure and hashes, then write a manifest.


## 1. One-time account preparation

Before the first run:

1. Open [`black-forest-labs/FLUX.2-dev`](https://huggingface.co/black-forest-labs/FLUX.2-dev), accept its access conditions, and create a Hugging Face token with read permission. The Base 4B repository is public, but the official standalone `ae.safetensors` is served from this gated repository.
2. Open [Kaggle API settings](https://www.kaggle.com/settings/api) and generate an API token. Copy the token value, not a legacy `kaggle.json` document.

In the Colab web interface, open the key icon in the left sidebar and create two secrets named exactly `HF_TOKEN` and `KAGGLE_API_TOKEN`. Enable **Notebook access** for both secrets. The notebook reads them with `google.colab.userdata`; secret values are not stored in notebook cells or outputs.


## 2. Download-only dependencies

These versions are intentionally compatible with the project's Transformers stack. The cell does not install or replace PyTorch, Diffusers, Transformers, or CUDA packages.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

PINNED_DOWNLOAD_PACKAGES = {
    "huggingface-hub": "0.36.2",
    "kagglehub": "1.0.1",
}

requirements = []
for distribution, expected_version in PINNED_DOWNLOAD_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        requirement = f"{distribution}=={expected_version}"
        if distribution == "huggingface-hub":
            requirement = f"huggingface_hub[hf_xet]=={expected_version}"
        requirements.append(requirement)

if requirements:
    print("Installing:", " ".join(requirements))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            *requirements,
        ],
        check=True,
    )
else:
    print("All download dependencies already match.")

{
    name: importlib.metadata.version(name)
    for name in PINNED_DOWNLOAD_PACKAGES
}


## 3. Mount Google Drive

Running this cell in Colab Web opens Google's Drive authorization flow. Approve it before continuing. This is the notebook's only Drive mount cell.


In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "diffusion-arc" / "models"

drive.mount(str(DRIVE_MOUNT), force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

drive_usage = shutil.disk_usage(DRIVE_ROOT)
assert drive_usage.free >= 32 * 10**9, (
    f"At least 32 GB free space is required; found {drive_usage.free / 10**9:.1f} GB"
)

{
    "drive_root": str(DRIVE_ROOT),
    "free_space_GB": round(drive_usage.free / 10**9, 1),
}


## 4. Pinned model identifiers and destinations

The FLUX revision matches the revision used by this repository's Diffusers VAE smoke test. The raw BFL transformer and both VAE formats are checked against the SHA-256 values pinned by Diffusion-ARC. NVARC is pinned to Kaggle version 1.


In [ ]:
FLUX_REPO_ID = "black-forest-labs/FLUX.2-klein-base-4B"
FLUX_REVISION = "8c44a2fbef88fae175da65df054db0f901aa9747"
STANDALONE_VAE_REPO_ID = "black-forest-labs/FLUX.2-dev"
STANDALONE_VAE_REVISION = "main"
NVARC_HANDLE = "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"

FLUX_DIR = DRIVE_ROOT / "flux2-klein-base-4b"
STANDALONE_VAE_DIR = DRIVE_ROOT / "flux2-standalone-vae"
NVARC_DIR = DRIVE_ROOT / "nvarc-qwen3-4b-grids15-sft139"
MANIFEST_PATH = DRIVE_ROOT / "download-manifest.json"

EXPECTED_PROJECT_SHA256 = {
    "flux_transformer": "9c5fed22b76baea749d88fc2abe3ad53245e7b21a0d353a762665eea00043b92",
    "diffusers_vae": "ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04",
    "standalone_vae": "868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3",
}

EXPECTED_NVARC_FILE_SIZES = {
    "model-00001-of-00002.safetensors": 4_996_836_472,
    "model-00002-of-00002.safetensors": 2_270_397_024,
}

{
    "flux": str(FLUX_DIR),
    "standalone_vae": str(STANDALONE_VAE_DIR),
    "nvarc": str(NVARC_DIR),
    "manifest": str(MANIFEST_PATH),
}


## 5. Load tokens from Colab Secrets

This cell reads the two secrets configured in Colab's left sidebar and exposes them to the download libraries for the lifetime of the current runtime. It prints only the secret names, never their values.


In [ ]:
import os

from google.colab import userdata

def read_required_secret(name: str) -> str:
    try:
        value = userdata.get(name)
    except Exception as error:
        raise RuntimeError(
            f"Colab secret {name!r} is missing or unavailable. Add it under "
            "the key icon in the left sidebar and enable Notebook access."
        ) from error
    if not value or not value.strip():
        raise RuntimeError(f"Colab secret {name!r} is empty.")
    return value.strip()

SECRET_NAMES = ("HF_TOKEN", "KAGGLE_API_TOKEN")
for secret_name in SECRET_NAMES:
    os.environ[secret_name] = read_required_secret(secret_name)

print("Loaded Colab secrets:", ", ".join(SECRET_NAMES))


## 6. Download the complete FLUX snapshot

`snapshot_download` preserves the complete Hugging Face repository layout required by `Flux2KleinPipeline.from_pretrained(...)`. It resumes interrupted transfers and reuses files that already match the pinned revision. Expect roughly 23.7 GB.


In [ ]:
assert os.environ.get("HF_TOKEN"), "Load HF_TOKEN from Colab Secrets first."

from huggingface_hub import snapshot_download

FLUX_DIR.mkdir(parents=True, exist_ok=True)
flux_snapshot_path = Path(
    snapshot_download(
        repo_id=FLUX_REPO_ID,
        revision=FLUX_REVISION,
        local_dir=str(FLUX_DIR),
        token=os.environ["HF_TOKEN"],
        force_download=False,
    )
)
assert flux_snapshot_path.resolve() == FLUX_DIR.resolve(), flux_snapshot_path
print("Complete FLUX snapshot:", flux_snapshot_path)


## 7. Download the standalone BFL autoencoder

The complete snapshot contains the Diffusers-format VAE. Diffusion-ARC's strict vendored loader additionally requires the original BFL state-dict file named `ae.safetensors`. A 401 or 403 here normally means the FLUX.2-dev access conditions have not been accepted for the account that owns the token.


In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

STANDALONE_VAE_DIR.mkdir(parents=True, exist_ok=True)
try:
    standalone_vae_path = Path(
        hf_hub_download(
            repo_id=STANDALONE_VAE_REPO_ID,
            filename="ae.safetensors",
            revision=STANDALONE_VAE_REVISION,
            local_dir=str(STANDALONE_VAE_DIR),
            token=os.environ["HF_TOKEN"],
            force_download=False,
        )
    )
except (GatedRepoError, HfHubHTTPError) as error:
    raise PermissionError(
        "Could not download the official standalone VAE. Accept access at "
        "https://huggingface.co/black-forest-labs/FLUX.2-dev and rerun "
        "the token and VAE cells with a read token from the same account."
    ) from error

print("Standalone BFL VAE:", standalone_vae_path)


## 8. Download the pinned NVARC checkpoint from Kaggle

Kaggle version 1 contains two BF16 Safetensors shards plus the model and tokenizer configuration. `output_dir` writes directly into Drive. Rerunning with `force_download=False` keeps a complete existing download and lets KaggleHub recover its cache state.


In [ ]:
assert os.environ.get("KAGGLE_API_TOKEN"), (
    "Load KAGGLE_API_TOKEN from Colab Secrets first."
)

import kagglehub

NVARC_DIR.mkdir(parents=True, exist_ok=True)
nvarc_download_path = Path(
    kagglehub.model_download(
        NVARC_HANDLE,
        output_dir=str(NVARC_DIR),
        force_download=False,
    )
)
print("NVARC checkpoint:", nvarc_download_path)


## 9. Validate the downloaded structure

This fails early if a transfer stopped before all required components arrived. It also checks the exact published sizes of both NVARC weight shards.


In [ ]:
required_flux_entries = (
    "model_index.json",
    "scheduler",
    "text_encoder",
    "tokenizer",
    "transformer",
    "vae",
    "flux-2-klein-base-4b.safetensors",
)
missing_flux_entries = [
    name for name in required_flux_entries if not (FLUX_DIR / name).exists()
]
assert not missing_flux_entries, f"Incomplete FLUX snapshot: {missing_flux_entries}"

standalone_vae_path = STANDALONE_VAE_DIR / "ae.safetensors"
assert standalone_vae_path.is_file(), standalone_vae_path

required_nvarc_files = (
    "added_tokens.json",
    "config.json",
    "generation_config.json",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "model.safetensors.index.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
)
missing_nvarc_files = [
    name for name in required_nvarc_files if not (NVARC_DIR / name).is_file()
]
assert not missing_nvarc_files, f"Incomplete NVARC checkpoint: {missing_nvarc_files}"

for filename, expected_size in EXPECTED_NVARC_FILE_SIZES.items():
    actual_size = (NVARC_DIR / filename).stat().st_size
    assert actual_size == expected_size, (filename, expected_size, actual_size)

print("FLUX, both VAE formats, and NVARC have the required structure.")


## 10. Verify project checkpoint identities

The hashes below are the identities expected by Diffusion-ARC. This reads about 8.3 GB from Drive and can take several minutes, but it prevents a similarly named or partially corrupted file from becoming experiment input.


In [ ]:
import hashlib

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

project_checkpoint_paths = {
    "flux_transformer": FLUX_DIR / "flux-2-klein-base-4b.safetensors",
    "diffusers_vae": FLUX_DIR / "vae" / "diffusion_pytorch_model.safetensors",
    "standalone_vae": standalone_vae_path,
}

actual_project_sha256 = {}
for name, path in project_checkpoint_paths.items():
    print(f"Hashing {name}: {path}")
    actual_project_sha256[name] = sha256_file(path)
    assert actual_project_sha256[name] == EXPECTED_PROJECT_SHA256[name], (
        name,
        EXPECTED_PROJECT_SHA256[name],
        actual_project_sha256[name],
    )

actual_project_sha256


## 11. Write a reproducibility manifest

The manifest records source identifiers, resolved Hugging Face revisions, paths, sizes, hashes, and downloader versions. It contains no authentication material.


In [ ]:
import json
from datetime import datetime, timezone

from huggingface_hub import HfApi

def directory_size(path: Path) -> int:
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())

hf_api = HfApi(token=os.environ["HF_TOKEN"])
resolved_flux_revision = hf_api.model_info(
    FLUX_REPO_ID, revision=FLUX_REVISION
).sha
resolved_vae_revision = hf_api.model_info(
    STANDALONE_VAE_REPO_ID, revision=STANDALONE_VAE_REVISION
).sha

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "drive_root": str(DRIVE_ROOT),
    "downloaders": {
        name: importlib.metadata.version(name)
        for name in PINNED_DOWNLOAD_PACKAGES
    },
    "flux": {
        "repo_id": FLUX_REPO_ID,
        "requested_revision": FLUX_REVISION,
        "resolved_revision": resolved_flux_revision,
        "path": str(FLUX_DIR),
        "directory_bytes": directory_size(FLUX_DIR),
        "project_file_sha256": {
            name: actual_project_sha256[name]
            for name in ("flux_transformer", "diffusers_vae")
        },
    },
    "standalone_vae": {
        "repo_id": STANDALONE_VAE_REPO_ID,
        "requested_revision": STANDALONE_VAE_REVISION,
        "resolved_revision": resolved_vae_revision,
        "path": str(standalone_vae_path),
        "bytes": standalone_vae_path.stat().st_size,
        "sha256": actual_project_sha256["standalone_vae"],
    },
    "nvarc": {
        "kaggle_handle": NVARC_HANDLE,
        "version": 1,
        "path": str(NVARC_DIR),
        "directory_bytes": directory_size(NVARC_DIR),
        "weight_file_bytes": EXPECTED_NVARC_FILE_SIZES,
    },
}

temporary_manifest = MANIFEST_PATH.with_suffix(".json.tmp")
temporary_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
temporary_manifest.replace(MANIFEST_PATH)
print("Manifest written to:", MANIFEST_PATH)
manifest


## Paths for later test notebooks

Loading large models directly from mounted Drive is convenient but slower than Colab's local SSD. For repeated inference in one VM, copy the needed model directory or checkpoint to `/content/checkpoints` first, then load it locally. Keep Drive as the durable source of truth.


In [ ]:
model_paths = {
    "diffusers_flux_snapshot": str(FLUX_DIR),
    "raw_flux_transformer": str(
        FLUX_DIR / "flux-2-klein-base-4b.safetensors"
    ),
    "diffusers_vae": str(FLUX_DIR / "vae"),
    "standalone_bfl_vae": str(standalone_vae_path),
    "nvarc": str(NVARC_DIR),
    "manifest": str(MANIFEST_PATH),
}

for name, path in model_paths.items():
    print(f"{name}: {path}")

model_paths


## Exercise and answer scaffold

**Exercise:** after starting a new Colab VM, mount Drive and confirm that every durable model path still exists without rerunning a download. Predict which files would disappear if they had instead been stored under `/content`.

The answer scaffold below performs the persistent-path check without loading model weights.


In [ ]:
persistent_path_check = {
    name: Path(path).exists()
    for name, path in model_paths.items()
}
assert all(persistent_path_check.values()), persistent_path_check
persistent_path_check


## Common pitfalls

- **A Colab secret is missing or unavailable:** create `HF_TOKEN` and `KAGGLE_API_TOKEN` under the key icon in Colab's left sidebar, then enable notebook access for both. Do not paste tokens into a code cell or committed `.env` file.
- **Drive authorization fails:** confirm that the notebook is running at `colab.research.google.com`, reconnect the runtime, and rerun the single Drive mount cell.
- **Hugging Face returns 401 or 403:** accept the FLUX.2-dev access conditions while signed into the same account that owns the token.
- **Kaggle authentication fails:** generate the current API token from Kaggle settings and enter the token value, not the legacy JSON document.
- **A download is interrupted:** keep the same destination and rerun the affected cell. Do not use `force_download=True` unless intentionally replacing the stored files.
- **The strict VAE loader rejects the Diffusers weight:** use `flux2-standalone-vae/ae.safetensors`; `flux2-klein-base-4b/vae/diffusion_pytorch_model.safetensors` has different state-dict names.
- **Inference from Drive is slow:** copy the durable files to `/content` for the lifetime of the current VM, then load from `/content`.
